# 04 · Visualización, expresión diferencial y análisis posteriores
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/es/04_visualization_deg_downstream.ipynb)

**De Células a Clústeres: Introducción al Análisis de Single-Cell RNA-seq en Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Perú · 9 de noviembre de 2026  
Instructora: Dra. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Tiempo en el tutorial:** 11:35–12:05 (30 min)

**Al final de este notebook podrás:**
- Hacer UMAPs, violin plots, dot plots, matrix plots y heatmaps con calidad de publicación
- Graficar proporciones de tipos celulares
- Encontrar genes diferencialmente expresados (DEGs) entre tipos celulares y hacer un volcano plot
- Hacer análisis de sobrerrepresentación (MSigDB Hallmark) e inferencia de actividad de vías (PROGENy)
- Entender por qué las pruebas a nivel de célula **no** son apropiadas para comparar condiciones

> 🟢 Las secciones **Core** se ven en vivo. 🔎 Las secciones **Extra** son para estudiar después del tutorial.

## 0. Configuración 🟢
Ejecuta primero la celda de abajo (▶ o `Shift + Enter`). En Google Colab instala los paquetes (~1 min);
en tu computadora asume que ya ejecutaste `pip install -r requirements.txt`.

**Opcional:** cambia `USE_DRIVE = True` para guardar tus resultados en Google Drive y reutilizarlos en
los siguientes notebooks. Si lo dejas en `False`, cada notebook reconstruye lo que necesita automáticamente.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10" "decoupler>=2.1,<3"

USE_DRIVE = False  # True = guardar resultados en Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Funciones auxiliares del tutorial (descarga + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # ejecutando desde un clon local
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import decoupler as dc
import seaborn as sns
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

In [ ]:
adata = tut.get_checkpoint("annotated", WORKDIR)
order = ["CD4 T", "CD8 T", "NK", "B", "CD14 Mono", "FCGR3A Mono", "DC", "Platelet"]
adata.obs["cell_type"] = adata.obs["cell_type"].cat.reorder_categories(
    [c for c in order if c in adata.obs["cell_type"].cat.categories])
markers = {k: v for k, v in tut.PBMC_MARKERS.items() if k in adata.obs["cell_type"].cat.categories}
adata

## 1. UMAP, de muchas formas 🟢
Consejos: pon las etiquetas sobre los datos en vez de una leyenda, usa un mapa de color perceptualmente uniforme
(`viridis`, `magma`) para la expresión y dibuja encima las células que expresan el gen (`sort_order=True`, por defecto).

In [ ]:
sc.pl.umap(adata, color="cell_type", legend_loc="on data", legend_fontsize=9,
           legend_fontoutline=2, title="PBMC 3k cell types")

In [ ]:
sc.pl.umap(adata, color=["MS4A1", "CD14", "GNLY", "FCGR3A"], cmap="magma", vmax="p99",
           ncols=4, use_raw=True)

In [ ]:
# Resaltar una población
sc.pl.umap(adata, color="cell_type", groups=["CD14 Mono", "FCGR3A Mono"], title="Monocytes")
# ¿Dónde se concentran las células?
sc.tl.embedding_density(adata, basis="umap", groupby="cell_type")
sc.pl.embedding_density(adata, basis="umap", key="umap_density_cell_type",
                        group=["CD4 T", "B", "CD14 Mono"], ncols=3)

## 2. Expresión entre grupos 🟢
| Gráfico | Ideal para |
|---|---|
| **Violin / stacked violin** | Distribución completa de pocos genes |
| **Dot plot** | Muchos genes × grupos: % que expresa + media |
| **Matrix plot** | Solo expresión media, compacto |
| **Heatmap / tracks plot** | Células individuales, muestra heterogeneidad dentro de un grupo |

In [ ]:
sc.pl.violin(adata, ["LYZ", "CD79A", "NKG7"], groupby="cell_type", rotation=60, use_raw=True)

In [ ]:
sc.pl.stacked_violin(adata, markers, groupby="cell_type", swap_axes=False, dendrogram=False, use_raw=True)

In [ ]:
sc.pl.dotplot(adata, markers, groupby="cell_type", standard_scale="var", use_raw=True,
              save="_pbmc_markers.pdf")   # guardado en WORKDIR/figures/

In [ ]:
sc.pl.matrixplot(adata, markers, groupby="cell_type", standard_scale="var", cmap="Blues", use_raw=True)

In [ ]:
sc.pl.heatmap(adata, markers, groupby="cell_type", standard_scale="var", cmap="viridis",
              use_raw=True, swap_axes=True, figsize=(10, 6))

In [ ]:
sc.pl.tracksplot(adata, markers, groupby="cell_type", use_raw=True)

## 3. Proporciones de tipos celulares 🟢
Con una sola muestra solo podemos describir la composición. Para **comparar** proporciones entre condiciones se
necesitan varias réplicas biológicas por condición (notebook 05).

In [ ]:
props = adata.obs["cell_type"].value_counts(normalize=True).reindex(adata.obs["cell_type"].cat.categories)
colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"])) \
    if "cell_type_colors" in adata.uns else None
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={"width_ratios": [3, 1]})
props.mul(100).plot.bar(ax=axes[0], color=[colors[c] for c in props.index] if colors else None)
axes[0].set(ylabel="% of cells", xlabel="")
bottom = 0
for ct, p in props.items():
    axes[1].bar(0, p * 100, bottom=bottom, color=colors[ct] if colors else None, label=ct)
    bottom += p * 100
axes[1].set(xticks=[], ylabel="% of cells", title="PBMC 3k")
axes[1].legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
plt.tight_layout()

## 4. Genes diferencialmente expresados entre tipos celulares 🟢
Probamos cada tipo celular contra todas las demás células. `pts=True` también reporta la fracción de células que
expresan cada gen, lo que ayuda a elegir marcadores **específicos** (alto en el grupo, bajo en el resto).

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="cell_type", method="wilcoxon", pts=True)
sc.pl.rank_genes_groups(adata, n_genes=15, ncols=4, sharey=False, fontsize=8)

In [ ]:
deg = sc.get.rank_genes_groups_df(adata, group="CD14 Mono")
deg_sig = deg.query("pvals_adj < 0.05 and abs(logfoldchanges) > 1")
print(f"{len(deg_sig)} significant genes ({(deg_sig.logfoldchanges > 0).sum()} up, {(deg_sig.logfoldchanges < 0).sum()} down)")
deg_sig.head(10)

In [ ]:
def volcano(df, title, lfc=1, padj=0.05, n_label=10):
    d = df.copy()
    d["nlp"] = -np.log10(d["pvals_adj"].clip(lower=1e-300))
    sig = (d["pvals_adj"] < padj) & (d["logfoldchanges"].abs() > lfc)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(d.loc[~sig, "logfoldchanges"], d.loc[~sig, "nlp"], s=4, c="lightgray")
    ax.scatter(d.loc[sig, "logfoldchanges"], d.loc[sig, "nlp"], s=6,
               c=np.where(d.loc[sig, "logfoldchanges"] > 0, "#d62728", "#1f77b4"))
    for _, r in d[sig].sort_values("nlp", ascending=False).head(n_label).iterrows():
        ax.annotate(r["names"], (r["logfoldchanges"], r["nlp"]), fontsize=8)
    ax.axvline(lfc, ls="--", c="k", lw=0.5); ax.axvline(-lfc, ls="--", c="k", lw=0.5)
    ax.axhline(-np.log10(padj), ls="--", c="k", lw=0.5)
    ax.set(xlabel="log2 fold change", ylabel="-log10 adjusted p-value", title=title)

volcano(deg, "CD14 monocytes vs. all other cells")

In [ ]:
# Solo marcadores específicos: >50% del grupo y <20% del resto
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, max_out_group_fraction=0.2,
                               min_fold_change=1, key_added="rank_genes_filtered")
sc.pl.rank_genes_groups_dotplot(adata, key="rank_genes_filtered", n_genes=4, standard_scale="var")

⚠️ **Marcadores de clúster ≠ expresión diferencial entre condiciones.** La prueba de Wilcoxon trata cada célula
como una réplica independiente. Las células de una misma persona no son independientes, así que los p-valores se
vuelven diminutos y los falsos positivos se disparan al comparar, por ejemplo, pacientes vs. controles. Para condiciones,
suma los conteos por muestra y tipo celular (**pseudobulk**) y usa herramientas de RNA-seq bulk (DESeq2, edgeR). Eso
hacemos en el notebook 05.

## 5. Análisis posteriores: ¿qué significan los DEGs? 🟢
### 5.1 Análisis de sobrerrepresentación (ORA) con MSigDB Hallmark
¿Los principales marcadores de cada tipo celular están enriquecidos en procesos biológicos conocidos? ORA es una prueba
exacta de Fisher entre tu lista de genes y cada conjunto de genes. Usamos los 50 conjuntos **Hallmark** de MSigDB (vía OmniPath).

In [ ]:
hallmark = tut.cached_table("hallmark_human", lambda: dc.op.hallmark(organism="human"), WORKDIR)
print(hallmark["source"].nunique(), "gene sets")

all_deg = sc.get.rank_genes_groups_df(adata, group=None)
# matriz tipos celulares x genes con log fold changes positivos
lfc = (all_deg.query("pvals_adj < 0.05").pivot_table(index="group", columns="names",
       values="logfoldchanges", observed=True).fillna(0).clip(lower=0))
ora_es, ora_p = dc.mt.ora(lfc, net=hallmark, n_up=100, tmin=5)
top_sets = (-np.log10(ora_p.clip(lower=1e-30))).max().sort_values(ascending=False).head(12).index
fig, ax = plt.subplots(figsize=(7, 0.35 * len(top_sets) + 1.5))
sns.heatmap((-np.log10(ora_p[top_sets].clip(lower=1e-30))).T.rename(index=lambda x: x.replace("HALLMARK_", "")),
            cmap="Reds", ax=ax, cbar_kws={"label": "-log10 adj. p"})
ax.set(xlabel="", ylabel="", title="Hallmark ORA on cell type markers")
plt.show()

### 5.2 Actividad de vías por célula con PROGENy
**PROGENy** proporciona "huellas": genes que cambian cuando se activa una vía de señalización (JAK-STAT, NF-κB, TNF-α,
TGF-β...). Con un ULM estimamos un puntaje de actividad para cada célula. A diferencia de ORA, usa la **dirección y el
peso** de cada gen blanco.

In [ ]:
progeny = tut.cached_table("progeny_human", lambda: dc.op.progeny(organism="human", top=500), WORKDIR)
dc.mt.ulm(adata, net=progeny, raw=True)
pathway_acts = dc.pp.get_obsm(adata, "score_ulm")
sc.pl.matrixplot(pathway_acts, var_names=pathway_acts.var_names, groupby="cell_type",
                 cmap="RdBu_r", vcenter=0, colorbar_title="mean activity")

In [ ]:
sc.pl.umap(pathway_acts, color=[p for p in ["JAK-STAT", "NFkB", "TNFa"] if p in pathway_acts.var_names],
           cmap="RdBu_r", vcenter=0, ncols=3)

### 5.3 🔎 Extra: puntuar tu propia firma génica
Cualquier lista de genes (por ejemplo, de un artículo) se puede puntuar con `sc.tl.score_genes`: la expresión media de
la lista menos la media de un conjunto aleatorio de genes con expresión similar. Aquí, una firma de genes estimulados
por interferón, que será mucho más alta en las células tratadas con IFN-β del notebook 05.

In [ ]:
isg = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "LY6E", "IFITM1", "IRF7", "STAT1"]
sc.tl.score_genes(adata, [g for g in isg if g in adata.raw.var_names], score_name="ISG_score", use_raw=True)
sc.pl.violin(adata, "ISG_score", groupby="cell_type", rotation=60, stripplot=False)

### 💡 Te toca a ti
Repite el análisis de DEGs comparando **CD14 Mono** directamente contra **FCGR3A Mono** (en vez de contra todas
las células) y dibuja el volcano plot. *Pista:* `sc.tl.rank_genes_groups(..., groups=["CD14 Mono"], reference="FCGR3A Mono", key_added="mono")`
y `sc.get.rank_genes_groups_df(adata, group="CD14 Mono", key="mono")`.

In [ ]:
# Tu código aquí

<details><summary><b>Ver solución</b></summary>

```python
sc.tl.rank_genes_groups(adata, groupby="cell_type", groups=["CD14 Mono"], reference="FCGR3A Mono",
                        method="wilcoxon", key_added="mono")
volcano(sc.get.rank_genes_groups_df(adata, group="CD14 Mono", key="mono"), "CD14 vs FCGR3A monocytes")
```
</details>

## 6. Guardar

In [ ]:
for key in ["rank_genes_filtered", "mono"]:   # tablas con valores vacíos no se guardan en .h5ad
    adata.uns.pop(key, None)
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_final.h5ad")
print(os.listdir(f"{WORKDIR}/figures"))

---
## ✅ ¡Listo!
Último paso: varias muestras, dos condiciones y cómo se compara Python con R. Siguiente: **05 · Integración, condiciones y R vs Python**.